In [1]:
import warnings
warnings.filterwarnings('ignore')

(Tutorial_Protonation_Assessment)=
# Assessing protonation removal

*Inspecting a source-bound decision and applying its positional atom maps once.*

You may need to remove hydrogens that contradict a chosen reference-rule pH without losing their original indices. `msm.build.assess_protonation()` prepares a read-only decision. Its public `ProtonationAssessment` result separates inspection from execution; `apply()` reuses the same decision and prepares the complete subset before publication.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

See {func}`molsysmt.build.assess_protonation`, {class}`molsysmt.build.ProtonationAssessment` and {func}`molsysmt.build.reconcile_protonation`.
:::


## Basic usage

Start with three structures of the bundled deposited villin system. Repeating source structure 0 gives three local structure positions; these positions are separate from their labels. The reference-rule decision uses names and stored connectivity, not coordinate values.


In [2]:
import numpy as np
import molsysmt as msm
from molsysmt import pyunitwizard as puw

molsys = msm.convert(
    msm.systems['chicken villin HP35']['1vii.pdb'],
    to_form='molsysmt.MolSys', structure_indices=[0, 0, 0]
)
print('Source atoms and structures:', molsys.get(n_atoms=True), molsys.get(n_structures=True))

Source atoms and structures: 596 3


:::{note}
:class: dropdown

The {ref}`bundled systems catalog <user-foundations-entrance-demo-systems>` supplies this offline dataset.
:::

## Preparing the decision

At pH 12.0 the existing standard-name rules identify an N-terminal H3 and five lysine HZ3 atoms in this particular input. Explicit named states can use different supported H-name layouts: inspect the actual indices and references instead of assuming one hydrogen label for every vocabulary.


In [3]:
assessment = msm.build.assess_protonation(molsys, pH=12.0)
report = assessment.to_dict()
print('Decision:', report['status'])
print('Removed source indices:', report['removed_atom_indices'])
print('Removed atom names:', [entry[1] for entry in report['removed_atoms']])
print('Source remains:', molsys.get(n_atoms=True), 'atoms')

Decision: removal_required
Removed source indices: [ 10 129 400 489 511 548]
Removed atom names: ['H3', 'HZ3', 'HZ3', 'HZ3', 'HZ3', 'HZ3']
Source remains: 596 atoms


## Inspecting correspondence

The nested `atom_map` uses the existing `molsysmt.atom_map@1` schema. `source_to_output_atom_indices` has one entry per original atom, with -1 for a removed atom. `output_to_source_atom_indices` lists retained original indices in new atom order. Both are int64 arrays. IDs and names may repeat; neither is used to reconstruct this correspondence. Every source structure is retained in order.


In [4]:
atom_map = report['atom_map']
forward = atom_map['source_to_output_atom_indices']
retained = atom_map['output_to_source_atom_indices']
print('Map lengths:', len(forward), len(retained))
print('Inverse identity:', np.array_equal(forward[retained], np.arange(len(retained))))
print('Structure indices:', report['structure_indices'])

Map lengths: 596 590
Inverse identity: True
Structure indices: [0 1 2]


## Applying the decision

`apply()` does not call the hydrogen-reference calculation again. It verifies the source, prepares an independent native `MolSys` and projects all stored domains using ordinary removal. Changing a dictionary returned by `to_dict()` cannot change the prepared decision. Each successful assessment is single-use.


In [5]:
molsys_A = assessment.apply(molsys)
print('Source and result atoms:', molsys.get(n_atoms=True), molsys_A.get(n_atoms=True))
print('All retained structures:', molsys_A.get(n_structures=True))
print('Coordinates preserved:', np.array_equal(
    puw.get_value(molsys_A.structures.coordinates, to_unit='nm'),
    puw.get_value(molsys.structures.coordinates, to_unit='nm')[:, retained]
))
print('Execution:', assessment.to_dict()['execution']['status'])

Source and result atoms: 596 590
All retained structures: 3
Coordinates preserved: True
Execution: applied


## Distinguishing no change from unknown chemistry

At pH 7.4 this deposited system has no H to remove under the supported rules. An independent no-op still returns independent data. `partial_no_change` means that some scope was unassessed; `unassessed` means no group could be assessed. Neither is a certificate of a correct protonation state. `coverage` reports group indices, reasons, unassigned atoms and the chemical states outside the reference comparison.

`group_contexts` retains the actual group name and naming declaration for every group, including unassessed groups. Each record includes its assessment status, reason and the terminal/disulfide context used; empty groups have no connectivity context.


In [6]:
unchanged = msm.build.assess_protonation(molsys, pH=7.4)
print('Decision:', unchanged.to_dict()['status'])
molsys_B = unchanged.apply(molsys)
print('Independent result:', molsys_B is not molsys)
print('Atoms:', molsys_B.get(n_atoms=True))

Decision: no_change
Independent result: True
Atoms: 596


## Publishing in place

An explicit in-place request is supported on the original native `MolSys`. The result is None. The complete candidate is prepared before domain publication; preparation errors leave the source and unused decision intact. A true in-place no-op preserves domain identities. Old domain/result references become historical snapshots after an actual removal.


In [7]:
molsys_C = msm.copy(molsys)
pending = msm.build.assess_protonation(molsys_C, pH=12.0)
returned = pending.apply(molsys_C, in_place=True)
print('Returned:', returned)
print('Updated atoms:', molsys_C.get(n_atoms=True))

Returned: None
Updated atoms: 590


## Composing completed maps

You can compose these maps with maps from other supported preparation/export tools. Here the first step removes the terminal H at pH 9.8; a later pH 12.0 step removes the remaining five H selected by the rules. Compose the completed nested maps through the existing public tool; do not match labels between the systems.


In [8]:
first = msm.build.assess_protonation(molsys, pH=9.8)
molsys_A = first.apply(molsys)
second = msm.build.assess_protonation(molsys_A, pH=12.0)
molsys_B = second.apply(molsys_A)
composed = msm.basic.compose_atom_maps([first.to_dict()['atom_map'], second.to_dict()['atom_map']])
print('Atoms through both operations:', molsys_A.get(n_atoms=True), molsys_B.get(n_atoms=True))
print('Matches the direct retained map:', np.array_equal(
    composed['output_to_source_atom_indices'], retained
))

Atoms through both operations: 595 590
Matches the direct retained map: True


## Respecting source lifetime

A decision is bound to the original object/resource and its inventory, naming, chemistry and atom/structure axes. A different copy is a different source. An expired or changed source requires a new assessment. Coordinate edits on unchanged axes remain admissible: execution uses current retained coordinates. Callers must serialize source access; this is not a lock over arbitrary external mutation or an authentication mechanism.


In [9]:
guarded = msm.build.assess_protonation(molsys, pH=12.0)
try:
    guarded.apply(msm.copy(molsys))
except msm.ArgumentConflictError as error:
    print('A different source is rejected:', type(error).__name__)

ERROR: Arguments 'molecular_system' and 'assessment' conflict: The assessment belongs to a different or replaced source object/domain. Check the documented relationship between these arguments. Docs: https://www.uibcdf.org/MolSysMT (Hint: Check the documented relationship between these arguments. Docs: https://www.uibcdf.org/MolSysMT)


A different source is rejected: ArgumentConflictError


## Reading a modular file

The assessment also accepts supported source forms directly. For H5MSM 0.5 it projects the needed chemical/inventory context and axis metadata without reading coordinate series or named analysis payloads. Applying the decision then materializes the independent native result; that output and its extraction have their normal memory costs.


In [10]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    filename = str(Path(directory) / 'source.h5msm')
    msm.h5msm.write(molsys, filename)
    from_file = msm.build.assess_protonation(filename, pH=12.0)
    molsys_A = from_file.apply(filename)
    print('Result from file:', molsys_A.get(n_atoms=True), molsys_A.get(n_structures=True))

Result from file: 590 3


## Scientific scope and stored results

Only the existing fixed-pKa name-inventory method is implemented. Standard labels use those approximate thresholds; supported explicit Amber states constrain their side-chain inventory. Unknown groups, incompatible layouts and named-state termini remain unassessed. The source naming declaration is read as stored; a sequence parent is never substituted. See {ref}`group naming schemes <user-foundations-group-naming-schemes>`.

The reference chemical state supplies the connectivity context. Removal projects the shared atom axis in **every** stored state and structure, without changing the reference or structure-to-state associations. Other states are not certified by that comparison. No hydrogen addition, charge reassignment or force-field parameterization occurs. Use a fixed-state preparation workflow when chemical assignments, rather than names, must determine the intended inventory.

:::{warning}
These reference rules do not predict environmental pKa shifts. A deposited H may reflect an environment-specific state that differs from the rule. Inspect removed indices before applying a destructive edit; assessed coverage does not establish physical correctness, equilibration or docking readiness.
:::

Named interactions follow normal subset projection: observations involving removed participants are lost; surviving observations retain their original evaluated scope. A remapped analysis is a new result whose occurrence row indices belong to that result. Update cached selections against it rather than equating old/new row positions. Existing analyses are not recalculated. Mechanical arrays and current preparation history are projected, without an implicit new assignment.

A native assessment keeps metadata/maps and weak source references; it does not copy or hash the coordinate trajectory. Inventory/chemical fingerprints require linear metadata work, and execution can temporarily retain both source and candidate. The executable object is process-local; its detached report can be preserved with the application's typed-array serialization. It is not a new H5MSM domain. Reference datasets and producer versions remain in the report even without Ackredit; optional application-owned capture records the consulted criteria.

:::{seealso}
:class: dropdown

- {func}`molsysmt.build.reconcile_protonation` — assess and execute in one convenience call.
- {func}`molsysmt.basic.compose_atom_maps` — compose declared atom correspondences.
- {ref}`Tutorial_Get_expected_hydrogens` — naming and reference-rule bounds.
- {ref}`Tutorial_Fixed_State_Hydrogens` — materialize an explicitly prepared inventory.
:::
